In [1]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer

# 1. Simulate a real-world article dataset for summarization
data = {
    "article": [
        "The central bank announced a new policy regarding interest rates on Friday. The adjustments aim to curb rising inflation trends while supporting ongoing micro-economic stability frameworks.",
        "A breakthrough in quantum computing hardware was documented yesterday. Engineers successfully stabilized a 100-qubit matrix at near-room temperature thresholds, marking a massive leap for processing speeds."
    ],
    "summary": [
        "The central bank adjusted interest rates to curb inflation.",
        "Engineers successfully stabilized a 100-qubit quantum matrix."
    ]
}

df = pd.DataFrame(data)
raw_dataset = Dataset.from_pandas(df)

# 2. Load the standard T5-small tokenizer
model_checkpoint = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_t5_summarization(examples):
    inputs = ["summarize: " + doc for doc in examples["article"]]

    # Tokenize the source texts
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")

    # Tokenize the target summaries (labels) using the target tokenizer context manager
    labels = tokenizer(text_target=examples["summary"], max_length=32, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Map the preprocessing across the partitions
print("Preprocessing dataset for T5 summarization...")
tokenized_dataset = raw_dataset.map(preprocess_t5_summarization, batched=True)
print("Dataset successfully structured!")
print("Sample Input Token IDs:", tokenized_dataset[0]["input_ids"][:10])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Preprocessing dataset for T5 summarization...


Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Dataset successfully structured!
Sample Input Token IDs: [21603, 10, 37, 2069, 2137, 2162, 3, 9, 126, 1291]
